In [0]:
# 03 - Gold Aggregation


PROJECT_CATALOG = "demoworkspacejoby"

SILVER_SCHEMA = "fraud_silver"
GOLD_SCHEMA = "fraud_gold"

SILVER_DB = f"{PROJECT_CATALOG}.{SILVER_SCHEMA}"
GOLD_DB = f"{PROJECT_CATALOG}.{GOLD_SCHEMA}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_DB}")

DataFrame[]

In [0]:
from pyspark.sql.functions import (
    col, count, countDistinct, sum, avg, when, lit, desc,
    round as spark_round, weekofyear, date_trunc, to_date,
    datediff, lag, percentile_approx, stddev,
    min as spark_min, max as spark_max,
    year, month, date_format
)

from pyspark.sql.window import Window

txn = spark.table(f"{SILVER_DB}.transactions")
users = spark.table(f"{SILVER_DB}.users")
cards = spark.table(f"{SILVER_DB}.cards")

print("Silver transactions:")
txn.printSchema()

display(txn.limit(5))

Silver transactions:
root
 |-- mcc: string (nullable = true)
 |-- id: long (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: long (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- errors: string (nullable = true)
 |-- transaction_date: timestamp (nullable = true)
 |-- mcc_description: string (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- transaction_year: integer (nullable = true)
 |-- transaction_month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- time_of_day: string (nullable = true)



mcc,id,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,transaction_date,mcc_description,is_fraud,transaction_year,transaction_month,day_of_week,day_name,hour_of_day,time_of_day
5541,7475350,114,3398,64.0,Swipe Transaction,61195,North Hollywood,CA,91606.0,,2010-01-01T00:38:00.000Z,Service Stations,0,2010,1,6,Friday,0,Night
5411,7475453,1313,4183,2.77,Swipe Transaction,61746,Sutersville,PA,15083.0,,2010-01-01T02:57:00.000Z,"Grocery Stores, Supermarkets",0,2010,1,6,Friday,2,Night
3780,7475464,957,4532,14.21,Swipe Transaction,44795,Marysville,OH,43040.0,,2010-01-01T03:13:00.000Z,Computer Network Services,0,2010,1,6,Friday,3,Night
7538,7475498,1736,113,55.79,Swipe Transaction,24823,Brooklyn,NY,11211.0,,2010-01-01T04:27:00.000Z,Automotive Service Shops,0,2010,1,6,Friday,4,Night
5912,7475528,1541,228,4.22,Swipe Transaction,61806,Clinton,TN,37716.0,,2010-01-01T05:16:00.000Z,Drug Stores and Pharmacies,0,2010,1,6,Friday,5,Night


In [0]:
# Gold Table 1: Fraud by day of week
gold_fraud_by_day_of_week = (
    txn
    .filter(col("is_fraud") == 1)
    .groupBy("day_of_week", "day_name")
    .agg(
        count("*").alias("fraud_count"),
        spark_round(avg("amount"), 2).alias("avg_fraud_amount"),
        spark_round(sum("amount"), 2).alias("total_fraud_amount")
    )
    .orderBy("day_of_week")
)

gold_fraud_by_day_of_week.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.fraud_by_day_of_week"
)

display(gold_fraud_by_day_of_week)

day_of_week,day_name,fraud_count,avg_fraud_amount,total_fraud_amount
1,Sunday,98,114.61,11231.59
2,Monday,71,101.2,7184.95
3,Tuesday,64,123.77,7921.06
4,Wednesday,11,-22.51,-247.62
5,Thursday,73,139.95,10216.55
6,Friday,106,136.3,14447.45
7,Saturday,12,100.68,1208.2


In [0]:
# Gold Table 2: Daily fraud rate trend
gold_fraud_rate_trend = (
    txn
    .withColumn("txn_date", to_date("transaction_date"))
    .groupBy("txn_date")
    .agg(
        count("*").alias("total_transactions"),
        count(when(col("is_fraud") == 1, 1)).alias("fraud_transactions"),
        spark_round(
            sum(when(col("is_fraud") == 1, col("amount")).otherwise(0)),
            2
        ).alias("fraud_amount")
    )
    .withColumn(
        "fraud_rate_percent",
        spark_round(col("fraud_transactions") / col("total_transactions") * 100, 4)
    )
    .orderBy("txn_date")
)

gold_fraud_rate_trend.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.fraud_rate_trend"
)

display(gold_fraud_rate_trend.limit(10))

txn_date,total_transactions,fraud_transactions,fraud_amount,fraud_rate_percent
2010-01-01,3463,1,0.19,0.0289
2010-01-02,2989,0,0.0,0.0
2010-01-03,3311,1,339.0,0.0302
2010-01-04,3244,2,11.64,0.0617
2010-01-05,3330,1,8.76,0.03
2010-01-06,3365,0,0.0,0.0
2010-01-07,3346,2,-48.46,0.0598
2010-01-08,3016,4,383.24,0.1326
2010-01-09,3102,1,23.1,0.0322
2010-01-10,3416,5,530.01,0.1464


In [0]:
# Gold Table 3: Top fraudulent users
gold_top_fraudulent_users = (
    txn
    .filter(col("is_fraud") == 1)
    .groupBy("client_id")
    .agg(
        count("*").alias("fraud_count"),
        spark_round(sum("amount"), 2).alias("total_fraud_amount"),
        spark_round(avg("amount"), 2).alias("avg_fraud_amount"),
        spark_min("transaction_date").alias("first_fraud_date"),
        spark_max("transaction_date").alias("last_fraud_date")
    )
    .join(
        users.select(
            col("id").alias("user_id"),
            "current_age",
            "gender",
            "credit_score",
            "yearly_income",
            "total_debt"
        ),
        col("client_id") == col("user_id"),
        "left"
    )
    .drop("user_id")
    .orderBy(desc("fraud_count"))
)

gold_top_fraudulent_users.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.top_fraudulent_users"
)

display(gold_top_fraudulent_users.limit(10))

client_id,fraud_count,total_fraud_amount,avg_fraud_amount,first_fraud_date,last_fraud_date,current_age,gender,credit_score,yearly_income,total_debt
1369,15,2085.8,139.05,2010-02-25T10:20:00.000Z,2010-02-28T11:57:00.000Z,59,Female,850,35854.0,49101.0
1511,14,1258.14,89.87,2010-02-07T07:05:00.000Z,2010-02-13T07:42:00.000Z,43,Male,719,48279.0,114254.0
1242,13,1308.91,100.69,2010-03-05T05:16:00.000Z,2010-03-08T03:46:00.000Z,52,Female,641,30509.0,56953.0
1519,12,1336.24,111.35,2010-03-02T05:49:00.000Z,2010-03-07T07:43:00.000Z,41,Female,849,40012.0,59472.0
1644,12,961.35,80.11,2010-01-05T02:34:00.000Z,2010-02-15T04:23:00.000Z,83,Male,686,31943.0,1248.0
1843,11,1414.37,128.58,2010-02-09T06:05:00.000Z,2010-02-13T09:07:00.000Z,89,Male,760,38761.0,69.0
1694,11,1387.64,126.15,2010-02-18T13:15:00.000Z,2010-02-23T14:06:00.000Z,83,Female,726,74753.0,1585.0
1106,10,1257.23,125.72,2010-02-19T06:00:00.000Z,2010-02-22T09:29:00.000Z,62,Female,699,20783.0,23623.0
1210,10,693.25,69.33,2010-03-05T06:06:00.000Z,2010-03-08T13:51:00.000Z,32,Male,703,34531.0,60156.0
1733,10,1051.51,105.15,2010-02-21T06:04:00.000Z,2010-02-24T03:13:00.000Z,47,Female,498,26130.0,54290.0


In [0]:
# Gold Table 4: User transaction amount spikes
weekly_window = Window.partitionBy("client_id").orderBy("txn_week")

user_weekly_amounts = (
    txn
    .withColumn("txn_week", date_trunc("week", "transaction_date"))
    .groupBy("client_id", "txn_week")
    .agg(
        count("*").alias("weekly_transaction_count"),
        spark_round(sum("amount"), 2).alias("weekly_total_amount"),
        spark_round(avg("amount"), 2).alias("weekly_avg_amount")
    )
)

gold_user_amount_spikes = (
    user_weekly_amounts
    .withColumn(
        "previous_4_week_avg_amount",
        avg("weekly_avg_amount").over(weekly_window.rowsBetween(-4, -1))
    )
    .filter(col("previous_4_week_avg_amount").isNotNull())
    .withColumn(
        "spike_ratio",
        spark_round(col("weekly_avg_amount") / col("previous_4_week_avg_amount"), 2)
    )
    .filter(col("spike_ratio") >= 3)
    .orderBy(desc("spike_ratio"))
)

gold_user_amount_spikes.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.user_amount_spikes"
)

display(gold_user_amount_spikes.limit(10))

client_id,txn_week,weekly_transaction_count,weekly_total_amount,weekly_avg_amount,previous_4_week_avg_amount,spike_ratio
900,2010-01-04T00:00:00.000Z,8,304.8,38.1,0.44,86.59
631,2010-01-11T00:00:00.000Z,13,527.74,40.6,0.9150000000000009,44.37
1508,2010-01-04T00:00:00.000Z,8,226.48,28.31,0.7,40.44
65,2010-01-04T00:00:00.000Z,13,583.98,44.92,1.63,27.56
212,2010-01-04T00:00:00.000Z,16,840.31,52.52,1.98,26.53
1452,2010-03-08T00:00:00.000Z,1,1321.98,1321.98,50.6125,26.12
980,2010-01-04T00:00:00.000Z,18,615.18,34.18,1.53,22.34
1434,2010-01-04T00:00:00.000Z,12,441.36,36.78,1.67,22.02
1300,2010-01-11T00:00:00.000Z,22,484.41,22.02,1.0150000000000006,21.69
1649,2010-01-04T00:00:00.000Z,2,147.02,73.51,6.16,11.93


In [0]:
# Gold Table 5: Fraud rate by merchant category
gold_fraud_rate_by_merchant_category = (
    txn
    .groupBy("mcc", "mcc_description")
    .agg(
        count("*").alias("total_transactions"),
        count(when(col("is_fraud") == 1, 1)).alias("fraud_count"),
        spark_round(
            sum(when(col("is_fraud") == 1, col("amount")).otherwise(0)),
            2
        ).alias("total_fraud_amount")
    )
    .withColumn(
        "fraud_rate_percent",
        spark_round(col("fraud_count") / col("total_transactions") * 100, 4)
    )
    .orderBy(desc("fraud_rate_percent"))
)

gold_fraud_rate_by_merchant_category.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.fraud_rate_by_merchant_category"
)

display(gold_fraud_rate_by_merchant_category.limit(10))

mcc,mcc_description,total_transactions,fraud_count,total_fraud_amount,fraud_rate_percent
4411,Cruise Lines,17,8,12651.96,47.0588
5733,Music Stores - Musical Instruments,8,2,38.37,25.0
5045,"Computers, Computer Peripheral Equipment",55,12,1522.96,21.8182
5533,Automotive Parts and Accessories Stores,18,3,1947.55,16.6667
5732,Electronics Stores,125,18,2420.66,14.4
3005,Miscellaneous Metal Fabrication,7,1,527.73,14.2857
5712,"Furniture, Home Furnishings, and Equipment Stores",79,11,2088.25,13.9241
4131,Bus Lines,45,6,187.6,13.3333
5816,Digital Goods - Games,55,7,859.41,12.7273
3144,Floor Covering Stores,8,1,366.06,12.5


In [0]:
# Gold Table 6: Merchants with high fraud volume
total_txn_count = txn.count()
fraud_txn_count = txn.filter(col("is_fraud") == 1).count()
global_fraud_rate = fraud_txn_count / total_txn_count * 100

merchant_fraud = (
    txn
    .groupBy("merchant_id", "merchant_city", "merchant_state", "mcc", "mcc_description")
    .agg(
        count("*").alias("total_transactions"),
        count(when(col("is_fraud") == 1, 1)).alias("fraud_count"),
        spark_round(
            sum(when(col("is_fraud") == 1, col("amount")).otherwise(0)),
            2
        ).alias("total_fraud_amount")
    )
    .withColumn(
        "fraud_rate_percent",
        spark_round(col("fraud_count") / col("total_transactions") * 100, 4)
    )
)

gold_merchants_high_fraud = (
    merchant_fraud
    .filter(col("fraud_count") > 0)
    .withColumn("global_fraud_rate_percent", lit(round(global_fraud_rate, 4)))
    .withColumn(
        "is_high_fraud_merchant",
        col("fraud_rate_percent") > lit(global_fraud_rate * 2)
    )
    .orderBy(desc("fraud_count"), desc("total_fraud_amount"))
)

gold_merchants_high_fraud.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.merchants_high_fraud"
)

display(gold_merchants_high_fraud.limit(10))

merchant_id,merchant_city,merchant_state,mcc,mcc_description,total_transactions,fraud_count,total_fraud_amount,fraud_rate_percent,global_fraud_rate_percent,is_high_fraud_merchant
60569,ONLINE,,5300,Wholesale Clubs,63,47,5358.75,74.6032,0.1977,true
32858,ONLINE,,5311,Department Stores,29,17,1516.74,58.6207,0.1977,true
34490,ONLINE,,5719,Miscellaneous Home Furnishing Stores,19,14,1456.87,73.6842,0.1977,true
84960,Port au Prince,Haiti,5310,Discount Stores,15,11,1087.31,73.3333,0.1977,true
47399,ONLINE,,5815,"Digital Goods - Media, Books, Apps",366,10,351.0,2.7322,0.1977,true
76639,ONLINE,,5732,Electronics Stores,13,8,1184.16,61.5385,0.1977,true
57133,ONLINE,,3730,Ship Chandlers,12,8,1015.99,66.6667,0.1977,true
41184,ONLINE,,5310,Discount Stores,10,8,752.96,80.0,0.1977,true
94123,ONLINE,,5310,Discount Stores,11,7,766.04,63.6364,0.1977,true
83271,ONLINE,,4214,Motor Freight Carriers and Trucking,11,7,64.84,63.6364,0.1977,true


In [0]:
# Gold Table 7: Fraud by time of day
gold_fraud_by_time_of_day = (
    txn
    .groupBy("time_of_day", "hour_of_day")
    .agg(
        count("*").alias("total_transactions"),
        count(when(col("is_fraud") == 1, 1)).alias("fraud_count"),
        spark_round(avg(when(col("is_fraud") == 1, col("amount"))), 2).alias("avg_fraud_amount"),
        spark_round(
            sum(when(col("is_fraud") == 1, col("amount")).otherwise(0)),
            2
        ).alias("total_fraud_amount")
    )
    .withColumn(
        "fraud_rate_percent",
        spark_round(col("fraud_count") / col("total_transactions") * 100, 4)
    )
    .orderBy("hour_of_day")
)

gold_fraud_by_time_of_day.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.fraud_by_time_of_day"
)

display(gold_fraud_by_time_of_day)

time_of_day,hour_of_day,total_transactions,fraud_count,avg_fraud_amount,total_fraud_amount,fraud_rate_percent
Night,0,2285,3,-104.39,-313.17,0.1313
Night,1,1955,9,58.79,529.07,0.4604
Night,2,1883,5,330.29,1651.47,0.2655
Night,3,1784,8,7.03,56.21,0.4484
Night,4,1909,11,-2.59,-28.45,0.5762
Night,5,3121,21,103.16,2166.45,0.6729
Morning,6,12362,50,121.8,6090.11,0.4045
Morning,7,14935,37,119.13,4407.76,0.2477
Morning,8,14343,33,146.49,4834.13,0.2301
Morning,9,14200,39,119.33,4653.74,0.2746


In [0]:
# Gold Table 8: Average amount — fraud vs legitimate
gold_avg_amount_fraud_vs_legit = (
    txn
    .groupBy("is_fraud")
    .agg(
        count("*").alias("transaction_count"),
        spark_round(avg("amount"), 2).alias("avg_amount"),
        spark_round(sum("amount"), 2).alias("total_amount"),
        spark_round(percentile_approx("amount", 0.5), 2).alias("median_amount"),
        spark_round(spark_min("amount"), 2).alias("min_amount"),
        spark_round(spark_max("amount"), 2).alias("max_amount"),
        spark_round(stddev("amount"), 2).alias("stddev_amount")
    )
    .withColumn(
        "transaction_type",
        when(col("is_fraud") == 1, "Fraud").otherwise("Legitimate")
    )
)

gold_avg_amount_fraud_vs_legit.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.avg_amount_fraud_vs_legit"
)

display(gold_avg_amount_fraud_vs_legit)

is_fraud,transaction_count,avg_amount,total_amount,median_amount,min_amount,max_amount,stddev_amount,transaction_type
0,219648,43.34,9520402.11,29.28,-500.0,4004.73,83.45,Legitimate
1,435,119.45,51962.18,69.7,-487.0,2529.39,264.8,Fraud


In [0]:
# Gold Table 9: Top fraud amount by merchant category
gold_top_fraud_amount_by_category = (
    txn
    .filter(col("is_fraud") == 1)
    .groupBy("mcc", "mcc_description")
    .agg(
        count("*").alias("fraud_count"),
        spark_round(sum("amount"), 2).alias("total_fraud_amount"),
        spark_round(avg("amount"), 2).alias("avg_fraud_amount")
    )
    .orderBy(desc("total_fraud_amount"))
)

gold_top_fraud_amount_by_category.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.top_fraud_amount_by_category"
)

display(gold_top_fraud_amount_by_category.limit(10))

mcc,mcc_description,fraud_count,total_fraud_amount,avg_fraud_amount
4411,Cruise Lines,8,12651.96,1581.5
5311,Department Stores,74,7894.47,106.68
5300,Wholesale Clubs,47,5358.75,114.02
5310,Discount Stores,41,3731.3,91.01
5732,Electronics Stores,18,2420.66,134.48
5712,"Furniture, Home Furnishings, and Equipment Stores",11,2088.25,189.84
5533,Automotive Parts and Accessories Stores,3,1947.55,649.18
5719,Miscellaneous Home Furnishing Stores,18,1641.88,91.22
5045,"Computers, Computer Peripheral Equipment",12,1522.96,126.91
4722,Travel Agencies,9,1127.82,125.31


In [0]:
# Gold Table 10: Daily fraud losses
gold_daily_fraud_losses = (
    txn
    .filter(col("is_fraud") == 1)
    .withColumn("txn_date", to_date("transaction_date"))
    .groupBy("txn_date")
    .agg(
        count("*").alias("fraud_count"),
        spark_round(sum("amount"), 2).alias("daily_fraud_loss"),
        spark_round(avg("amount"), 2).alias("avg_fraud_amount")
    )
    .orderBy("txn_date")
)

gold_daily_fraud_losses.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.daily_fraud_losses"
)

display(gold_daily_fraud_losses.limit(10))

txn_date,fraud_count,daily_fraud_loss,avg_fraud_amount
2010-01-01,1,0.19,0.19
2010-01-03,1,339.0,339.0
2010-01-04,2,11.64,5.82
2010-01-05,1,8.76,8.76
2010-01-07,2,-48.46,-24.23
2010-01-08,4,383.24,95.81
2010-01-09,1,23.1,23.1
2010-01-10,5,530.01,106.0
2010-01-11,2,302.7,151.35
2010-01-12,1,1014.41,1014.41


In [0]:
# Gold Table 11: Weekly unique fraud users
gold_weekly_unique_fraud_users = (
    txn
    .filter(col("is_fraud") == 1)
    .withColumn("txn_week", date_trunc("week", "transaction_date"))
    .groupBy("txn_week")
    .agg(
        countDistinct("client_id").alias("unique_fraud_users"),
        count("*").alias("total_fraud_transactions"),
        spark_round(sum("amount"), 2).alias("weekly_fraud_amount")
    )
    .orderBy("txn_week")
)

gold_weekly_unique_fraud_users.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.weekly_unique_fraud_users"
)

display(gold_weekly_unique_fraud_users.limit(10))

txn_week,unique_fraud_users,total_fraud_transactions,weekly_fraud_amount
2009-12-28T00:00:00.000Z,1,2,339.19
2010-01-04T00:00:00.000Z,5,15,908.29
2010-01-11T00:00:00.000Z,5,11,2802.15
2010-01-18T00:00:00.000Z,7,18,1606.5
2010-01-25T00:00:00.000Z,17,61,6597.44
2010-02-01T00:00:00.000Z,16,70,8716.37
2010-02-08T00:00:00.000Z,12,70,6118.08
2010-02-15T00:00:00.000Z,13,57,12682.69
2010-02-22T00:00:00.000Z,13,62,4970.97
2010-03-01T00:00:00.000Z,10,65,7418.77


In [0]:
# Gold Table 12: Monthly fraud spikes
gold_monthly_fraud_spikes = (
    txn
    .withColumn("txn_year", year("transaction_date"))
    .withColumn("txn_month", month("transaction_date"))
    .withColumn("year_month", date_format("transaction_date", "yyyy-MM"))
    .groupBy("txn_year", "txn_month", "year_month")
    .agg(
        count("*").alias("total_transactions"),
        count(when(col("is_fraud") == 1, 1)).alias("fraud_count"),
        spark_round(
            sum(when(col("is_fraud") == 1, col("amount")).otherwise(0)),
            2
        ).alias("monthly_fraud_amount")
    )
    .withColumn(
        "fraud_rate_percent",
        spark_round(col("fraud_count") / col("total_transactions") * 100, 4)
    )
    .orderBy("txn_year", "txn_month")
)

gold_monthly_fraud_spikes.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.monthly_fraud_spikes"
)

display(gold_monthly_fraud_spikes.limit(10))

txn_year,txn_month,year_month,total_transactions,fraud_count,monthly_fraud_amount,fraud_rate_percent
2010,1,2010-01,101209,107,12253.57,0.1057
2010,2,2010-02,93470,259,32488.11,0.2771
2010,3,2010-03,25404,69,7220.5,0.2716


In [0]:
# Gold Table 13: User behavior before vs after fraud
first_fraud_per_user = (
    txn
    .filter(col("is_fraud") == 1)
    .groupBy("client_id")
    .agg(spark_min("transaction_date").alias("first_fraud_date"))
)

txn_with_first_fraud = (
    txn
    .join(first_fraud_per_user, on="client_id", how="inner")
    .withColumn("days_from_first_fraud", datediff(col("transaction_date"), col("first_fraud_date")))
)

gold_user_behavior_around_fraud = (
    txn_with_first_fraud
    .withColumn(
        "period",
        when(col("days_from_first_fraud").between(-30, -1), "30d Before Fraud")
        .when(col("days_from_first_fraud") == 0, "Day of Fraud")
        .when(col("days_from_first_fraud").between(1, 30), "30d After Fraud")
    )
    .filter(col("period").isNotNull())
    .groupBy("period")
    .agg(
        count("*").alias("transaction_count"),
        countDistinct("client_id").alias("unique_users"),
        spark_round(avg("amount"), 2).alias("avg_amount"),
        spark_round(sum("amount"), 2).alias("total_amount"),
        count(when(col("is_fraud") == 1, 1)).alias("fraud_transaction_count")
    )
)

gold_user_behavior_around_fraud.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.user_behavior_around_fraud"
)

display(gold_user_behavior_around_fraud)

period,transaction_count,unique_users,avg_amount,total_amount,fraud_transaction_count
30d After Fraud,4584,62,50.4,231011.38,242
30d Before Fraud,4855,61,45.53,221045.49,0
Day of Fraud,437,62,91.61,40033.91,184


In [0]:
# Gold Table 14: Fraud by amount tier
gold_fraud_by_amount_tier = (
    txn
    .withColumn(
        "amount_tier",
        when(col("amount") < 0, "Negative/Refund")
        .when(col("amount") < 25, "$0-$25 Low")
        .when(col("amount") < 100, "$25-$100 Medium")
        .when(col("amount") < 500, "$100-$500 High")
        .when(col("amount") < 1000, "$500-$1K Very High")
        .otherwise("$1K+ Premium")
    )
    .groupBy("amount_tier")
    .agg(
        count("*").alias("total_transactions"),
        count(when(col("is_fraud") == 1, 1)).alias("fraud_count"),
        spark_round(sum("amount"), 2).alias("total_amount"),
        spark_round(
            sum(when(col("is_fraud") == 1, col("amount")).otherwise(0)),
            2
        ).alias("fraud_amount")
    )
    .withColumn(
        "fraud_rate_percent",
        spark_round(col("fraud_count") / col("total_transactions") * 100, 4)
    )
    .orderBy("amount_tier")
)

gold_fraud_by_amount_tier.write.mode("overwrite").saveAsTable(
    f"{GOLD_DB}.fraud_by_amount_tier"
)

display(gold_fraud_by_amount_tier)

amount_tier,total_transactions,fraud_count,total_amount,fraud_amount,fraud_rate_percent
$0-$25 Low,90034,83,912276.26,644.2,0.0922
$100-$500 High,25577,156,4103664.81,31350.49,0.6099
$1K+ Premium,163,7,226508.29,12306.09,4.2945
$25-$100 Medium,92304,160,5058623.52,9316.4,0.1733
$500-$1K Very High,612,7,425165.62,4580.0,1.1438
Negative/Refund,11393,22,-1153874.21,-6235.0,0.1931


In [0]:
# Final Gold validation
gold_tables = [
    "fraud_by_day_of_week",
    "fraud_rate_trend",
    "top_fraudulent_users",
    "user_amount_spikes",
    "fraud_rate_by_merchant_category",
    "merchants_high_fraud",
    "fraud_by_time_of_day",
    "avg_amount_fraud_vs_legit",
    "top_fraud_amount_by_category",
    "daily_fraud_losses",
    "weekly_unique_fraud_users",
    "monthly_fraud_spikes",
    "user_behavior_around_fraud",
    "fraud_by_amount_tier"
]

print("=" * 70)
print("GOLD AGGREGATION COMPLETE")
print("=" * 70)

for table_name in gold_tables:
    full_table_name = f"{GOLD_DB}.{table_name}"
    print(f"Counting {full_table_name} ...")
    row_count = spark.table(full_table_name).count()
    print(f"{full_table_name}: {row_count:,} rows")

print("=" * 70)
print(f"All {len(gold_tables)} gold tables are ready for dashboarding.")

GOLD AGGREGATION COMPLETE
Counting demoworkspacejoby.fraud_gold.fraud_by_day_of_week ...
demoworkspacejoby.fraud_gold.fraud_by_day_of_week: 7 rows
Counting demoworkspacejoby.fraud_gold.fraud_rate_trend ...
demoworkspacejoby.fraud_gold.fraud_rate_trend: 67 rows
Counting demoworkspacejoby.fraud_gold.top_fraudulent_users ...
demoworkspacejoby.fraud_gold.top_fraudulent_users: 62 rows
Counting demoworkspacejoby.fraud_gold.user_amount_spikes ...
demoworkspacejoby.fraud_gold.user_amount_spikes: 220 rows
Counting demoworkspacejoby.fraud_gold.fraud_rate_by_merchant_category ...
demoworkspacejoby.fraud_gold.fraud_rate_by_merchant_category: 109 rows
Counting demoworkspacejoby.fraud_gold.merchants_high_fraud ...
demoworkspacejoby.fraud_gold.merchants_high_fraud: 121 rows
Counting demoworkspacejoby.fraud_gold.fraud_by_time_of_day ...
demoworkspacejoby.fraud_gold.fraud_by_time_of_day: 24 rows
Counting demoworkspacejoby.fraud_gold.avg_amount_fraud_vs_legit ...
demoworkspacejoby.fraud_gold.avg_amount_